In [14]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import scipy.stats as sst
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.feature_selection import f_classif, chi2, SelectKBest, SelectPercentile
from collections import Counter
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, PrecisionRecallDisplay, precision_recall_curve, roc_curve, RocCurveDisplay, roc_auc_score, average_precision_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBClassifier, XGBRFClassifier
from sklearn.ensemble import StackingClassifier, AdaBoostClassifier, BaggingClassifier, ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, VotingClassifier, RandomForestClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import SGDClassifier

dataset = pd.read_csv('/content/drive/My Drive/Project5405/train_data.csv')
d2 = pd.read_csv('/content/drive/My Drive/Project5405/most_imp_feat_xgb.csv')


# imp = d2.iloc[:,1].tolist()[:50]    #Can increase decrease later
imp = d2.iloc[:,1].tolist()[:70]

total_pred_10 = []
accuracy_10 = []

# dataset = dataset.iloc[0:30000,:] ## remove this later

y = dataset.Label
X = dataset.drop(["Label","KIBA"],axis="columns")
X = dataset[imp]
X = MinMaxScaler().fit_transform(X)




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

# X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.5, stratify=y)#random_state=2)#,shuffle=True)#, 

for i in range(10):  ## change to 10 later

  X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.074699, stratify=y)#, random_state=42)#,shuffle=True)#, 

  y_train.replace({False: 0, True: 1}, inplace=True)
  y_test.replace({False: 0, True: 1}, inplace=True)


  xgb = XGBClassifier(use_label_encoder = False, max_depth = 20, learning_rate = 0.075, n_estimators = 450, scale_pos_weight = 1.5)# scale_pos_weight = 86324/23155)#, eval_metric = "error" #"logloss")#, max_depth =7)

  et = ExtraTreesClassifier(n_estimators=150, max_features="log2", min_samples_split=5)

  svm_sgd = CalibratedClassifierCV(SGDClassifier(max_iter=1000, tol=1e-3, loss = 'hinge'))


  models = [('xgb',xgb),('et',et),('svm_sgd',svm_sgd)]   ##best combination so far as of 1st december 10.45 am with pr@50 0.8530571992110454
  # models = [('et',et)]   ##test


  vc = VotingClassifier(estimators=models, voting="soft")

  vc.fit(X_train,y_train)


  pred_y = vc.predict(X_test)
  pred_y_prob = vc.predict_proba(X_test)[:,1]


  cm = confusion_matrix(y_test,pred_y)
  tn,fp,fn,tp = cm.ravel()
  Accuracy = (tp+tn)/(tp+tn+fp+fn)
  accuracy_10.append(Accuracy)

  # print("Accuracy: ",(tp+tn)/(tp+tn+fp+fn))
  # print("Precision: ", tp/(tp+fp))
  # print("Recall: ", tp/(tp+fn))
  # print("Sensitivity: ", tp/(tp+fn))
  # print("Specificity: ",tn/(tn+fp))
  # pr = tp/(tp+fp)
  # rc = tp/(tp+fn)
  # print("F1 score: ", (2*pr*rc)/(pr+rc))

  # PrecisionRecallDisplay.from_predictions(y_test,pred_y_prob,pos_label=1)

  precision, recall, thresholds = precision_recall_curve(y_test, pred_y_prob, pos_label=1)
  precision_inv = precision[::-1]
  recall_inv = recall[::-1]
  pr_res = np.interp(0.5, recall_inv, precision_inv)
  #pr_res = round(pr_res, 3)
  print("PR Score:" + str(pr_res))

  # plt.figure()
  # plt.axvline(0.5, 0, color="black", linestyle="dotted")#, label="Recall=0.5")
  # plt.axhline(pr_res, 0, color="black", linestyle="dotted", label= "PR@50Recall: {0}".format(pr_res))
  # lw = 2
  # plt.plot(recall, precision, color="darkorange", lw=lw)#, label= "PR@50Recall: {0}".format(pr_res), )
  # # plt.plot([0, 1], [0.5, 0.5], color="navy", lw=lw, linestyle="--")
  # plt.xlim([0.0, 1.1])
  # plt.ylim([0.0, 1.1])
  # plt.xlabel("Recall")
  # plt.ylabel("Precision")
  # plt.title("Precision Recall Curve")
  # plt.legend(loc="lower right")

  total_pred_10.append(pr_res)


# ConfusionMatrixDisplay.from_predictions(y_test,pred_y)   
# RocCurveDisplay.from_predictions(y_test,pred_y_prob,pos_label=1)





In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print(total_pred_10)
df = pd.DataFrame(data=total_pred_10)
df.index += 1
df.to_csv('/content/drive/My Drive/Project5405/10_pr@50rc.csv', index=True, header=False)

df = pd.DataFrame(data=accuracy_10)
df.index += 1
df.to_csv('/content/drive/My Drive/Project5405/10_acc.csv', index=True, header=False)